# PupilSense Left-Eye Reproduction (Stage 1)

Runs the released ResNet18/ResNet50 checkpoints over the EyeDentify dataset to reproduce the left-eye MAE/MAPE from *PupilSense* (Shah et al., ETRA '25).

**Before running:** in the Colab menu, go to Runtime > Change runtime type and select a GPU.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import sys
from pathlib import Path
import torch

PACKAGE_PARENT = "/content/drive/MyDrive/EyeBiomarkers"      # folder containing pupilsense_repro/
DATA_ROOT = Path("/content/drive/MyDrive/EyeBiomarkers/data/left_eyes_data/local_left_eyes_data")
WEIGHTS_DIR = Path("/content/drive/MyDrive/EyeBiomarkers/pre_trained_models")

sys.path.insert(0, PACKAGE_PARENT)
from pupilsense_repro.config import ReproConfig
from pupilsense_repro.runner import run_reproduction
from pupilsense_repro import plots

config = ReproConfig(
    data_root=DATA_ROOT,
    weights_dir=WEIGHTS_DIR,
    device="cuda" if torch.cuda.is_available() else "cpu",
    results_dir=Path("/content/results"),
    figures_dir=Path("/content/figures"),
)
print("device:", config.device)

In [ ]:
summary = run_reproduction(config, bases=("resnet18", "resnet50"))
summary

In [ ]:
import pandas as pd
config.figures_dir.mkdir(parents=True, exist_ok=True)
per_part = pd.read_csv(config.results_dir / "per_participant_mape.csv", index_col=0)
per_participant_by_base = {b: per_part[b].dropna() for b in per_part.columns}

plots.plot_per_participant_mape(per_participant_by_base, config.figures_dir / "per_participant_mape.png")
plots.plot_mape_histogram(per_participant_by_base, config.figures_dir / "mape_hist.png")
for base in ("resnet18", "resnet50"):
    preds = pd.read_csv(config.results_dir / f"predictions_{base}.csv")
    plots.plot_pred_vs_true(preds, config.figures_dir / f"pred_vs_true_{base}.png")
    plots.plot_diameter_over_frames(preds, participant_id=1, session_id=1,
                                    out_path=config.figures_dir / f"series_{base}.png")

from IPython.display import Image as IPyImage, display
display(IPyImage(str(config.figures_dir / "per_participant_mape.png")))

## Reading the results

`summary` (and `left_eye_metrics.csv` in `config.results_dir`) has one row per base architecture with `overall_mae`, `overall_mape`, and per-fold metrics; `per_participant_mape.csv` gives the per-participant MAPE distribution plotted above.

**Caveat:** the released weights are a single deployed checkpoint per eye/architecture, not per-fold models. Overall MAPE can be optimistic where the checkpoint trained on a participant that also appears in evaluation (train/test overlap). For a fairer comparison to the paper's left-eye numbers (ResNet18 ≈ 3.41%, ResNet50 ≈ 3.23%), look at the per-fold rows rather than the overall figure.